# Somo la 16 - Kuwasambaza Wakala Wanaoweza Kupanuliwa kwa Microsoft Foundry

Katika daftari hili unaunda **wakala wa msaada kwa wateja tayari kwa uzalishaji** kwa kampuni ya kubuni **Contoso**. Tofauti na masomo ya awali, lengo hapa si mzunguko wa hoja wa wakala — ni kila kitu kilicho *zirani* nacho kinachomfanya wakala awe salama kuendesha kwa wingi:

1. **Kupiga zana simu** — kuangalia maagizo na kuunda tiketi.
2. **RAG** — majibu ya sera kutoka kwenye msingi wa maarifa.
3. **Kumbukumbu** — kukumbuka mteja katika mzunguko wa mazungumzo.
4. **Uelekezaji wa mfano** — tuma maombi rahisi kwa mfano mdogo, magumu kwa mfano mkubwa.
5. **Kuhifadhi majibu** — hudumia maswali ya kurudiwa bila kupiga simu mfano.
6. **Idhini ya binadamu** — rufaa juu ya kiwango cha mitiathiri hutakiwa kusubiri idhini.
7. **Mlango wa tathmini** — seti ya majaribio ya mtandaoni inayozuia toleo baya.
8. **Uangalizi** — Ufuatiliaji wa OpenTelemetry karibu na kila ombi.

Kila sehemu ni huru na inaweza kuendeshwa. Soma kila mstari — msingi wa uzalishaji umebaki mdogo kwa makusudi.


## Usanidi

Kabla ya kuendesha daftari hii, hakikisha una:

1. **Mradi wa Microsoft Foundry** ulio na mfano wa mazungumzo uliotumika (mfano `gpt-4.1-mini`).
2. **Umeingia kwa kutumia Azure CLI** — run `az login` katika terminal yako.
3. **Weka vigezo vya mazingira vinavyohitajika:**
   - `AZURE_AI_PROJECT_ENDPOINT` — kiungo cha mradi wako wa Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jina la mfano uliotumika.

Sehemu ya RAG hutumia **Azure AI Search** wakati `AZURE_SEARCH_SERVICE_ENDPOINT` na `AZURE_SEARCH_API_KEY` vimewekwa, na inarudi kwa utafutaji wa kumbukumbu ili daftari lifanye kazi bila rasilimali ya Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Vifaa

Vifaa vya uzalishaji hufanya kazi halisi dhidi ya mifumo halisi. Hapa tunalinganisha hifadhidata ya maagizo na mfumo wa tiketi kwa kutumia kazi za kawaida za Python. Dekoreta ya `@tool` inawatambulisha kwa wakala.

Angalia `issue_refund` inatumia `approval_mode="always_require"` kwa marejesho ya fedha yanayozidi kikomo — huu ni mfumo wa mwanadamu katika mzunguko tunaoanzisha baadaye.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Hifadhidata ya Maarifa ya Sera

Maswali ya sera ("dirisha lako la kurudisha ni lini?") yanapaswa kujibiwa kutoka chanzo chenye mamlaka, si kumbukumbu ya mfano. Tunafunika hifadhidata ndogo kama chombo cha utafutaji.

Katika uzalishaji hii ni **Azure AI Search**; hapa tunatoa utafutaji wa maneno muhimu ndani ya kumbukumbu ili daftari liendeshwe popote, na kubadili kwa Azure AI Search moja kwa moja wakati vigezo vya mazingira vipo.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Kumbukumbu  

Mwakala wa msaada anayesahau anaongea nani ni mwakala mbaya wa msaada. Tunahifadhi duka dogo la wasifu kwa kila mteja na kuingiza muhtasari mfupi katika maelekezo ya mwakala. Katika uzalishaji hii ni huduma ya kumbukumbu (tazama Somo la 13); hapa kamusi inaonyesha muundo.  


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Upangaji wa Mfano na Kuhifadhi Majibu

Mikokotano miwili ya gharama imeunganishwa na mshughulikiaji moja wa maombi:

- **Upangaji**: kinadharia rahisi cha kutabiri huchagua kama ombi linahitaji mfano mdogo au mkubwa.
- **Kuhifadhi**: maswali yaliyorekebishwa hubadilishwa moja kwa moja kutoka kwenye hifadhi bila kuita mfano.

Kinadharia hapo ni rahisi kwa makusudi. Katika uzalishaji ungebaini dhidi ya trafiki na unaweza kuibadilisha na Model Router ya Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Wakala, Idhini ya Binadamu, na Ufuatiliaji

Sasa tunakusanya wakala kutoka kwa zana zilizo hapo juu na kufunika kila ombi katika eneo la OpenTelemetry. Kazi ya `handle_support_request` ni mteja wa ombi wa uzalishaji: cache → route → trace → run → cache.

Idhini ya binadamu inashughulikiwa na mfumo: kwa sababu `issue_refund` ipo `approval_mode="always_require"`, utekelezaji unasimama na kuonyesha ombi la idhini ambalo tunalitatua waziwazi.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Ganganuzi ya Tathmini

Hii ni mlango wa kuachilia kutoka somo: seti ya mtihani isiyo mtandaoni hupima wakala, na utekesaji unaendelea tu kama kiwango cha kupita kinavuka kizingiti. Mchambuzi hapa ni ukaguzi rahisi wa mduara wa maneno muhimu ili kuweka daftari kuwa huru; katika uzalishaji utatumia LLM kama hakimu au mchambuzi wa mfumo (angalau Somo la 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Kuunganisha Pamoja: Toleo la Kuigiza

Seli iliyo chini inaonyesha mzunguko mzima unaoelezewa na somo: endesha lango la tathmini, na "wasilisha" tu ikiwa litatangazwa. Huu ni muundo unaotakiwa kuendeshwa katika CI kabla ya kukuza toleo la wakala kwenda Huduma ya Wakala wa Foundry.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Muhtasari

Umebuni wakala wa msaada kwa wateja tayari kwa uzalishaji aliyebeba kila tatizo la uendeshaji limeunganishwa:

- **Vifaa, RAG, na kumbukumbu** vinampa wakala uwezo na muktadha.
- **Uelekezaji wa modeli na kuweka hifadhi** huhakikisha ucheleweshaji na gharama ziko chini ya udhibiti.
- **Idhini ya binadamu** inalinda hatua hatari kama kurejesha pesa kwa kiwango kikubwa.
- **Lango la tathmini** linazuia uzinduzi mbaya kabla hayajachapishwa.
- **Ufuatiliaji** hufanya kila ombi kuwa waonekana.

### Changamoto

Panua wakala huyu ili:

1. **Kusaidia modeli nyingi** — ongeza ngazi ya tatu ya "kusisitiza sababu" na elekeza kushikilia malalamiko/kikomo kwake.
2. **Ongeza lango la tathmini** — panua `TEST_CASES` ili kujumuisha hali za idhini za kurejesha pesa na hakikisha lango linakamata marejeleo.
3. **Ongeza uelekezaji wenye ufahamu wa gharama** — kufuatilia gharama inayokadiriwa kwa kila ombi (ndogo vs kubwa vs hifadhi) na chapisha ripoti ya gharama baada ya kundi la maswali mchanganyiko.

Katika somo lijalo utaenda kinyume na hii na kuendesha wakala mzima kwa mashine yako mwenyewe kwa kutumia Microsoft Foundry Local na Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
